In [ ]:
# Installing The Necessary Packages And Dependencies Required For Fine-Tuning The Wav2Vec2 Model Using The Svarah Dataset

!pip install -q numpy transformers datasets torchaudio jiwer huggingface_hub accelerate evaluate

In [ ]:
# Importing The Necessary Packages And Dependencies Required For Fine-Tuning The Wav2Vec2 Model Using The Svarah Dataset

import json
import numpy as np
import torch
import torchaudio

from dataclasses import dataclass
from datasets import Audio, load_dataset
from typing import Dict, List

from transformers import (
    Trainer,
    TrainingArguments,
    Wav2Vec2CTCTokenizer,
    Wav2Vec2FeatureExtractor,
    Wav2Vec2ForCTC,
    Wav2Vec2Processor,
)

RANDOM_SEED = 42
torch.manual_seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

if torch.cuda.is_available():
    print(f"The GPU Inference Model Detected : {torch.cuda.get_device_name(0)}")

else:
    print("No GPU Inference Model Detected")

print("")
print("The Necessary Packages And Dependencies Are Imported Successfully")

In [ ]:
# Authenticating The Hugging Face Hub To Store The Model Checkpoints And Access The Svarah Dataset

from huggingface_hub import login

login()

print("Successfully Authenticated With The Hugging Face Hub")

In [ ]:
# Loading The Svarah Speech Dataset From The Hugging Face Hub To The Hosted Cloud Environment

from datasets import Audio, load_dataset

print("Loading The Svarah Dataset From The Hugging Face Hub...")

dataset = load_dataset("ai4bharat/Svarah")

print("")
print("The Svarah Dataset Has Loaded Successfully From Hugging Face")

In [ ]:
# Casting The Audio Column To A Unified Sampling Rate Of 16KHz And Splitting The Dataset Into Training, Testing, And Validation Sets

dataset = dataset.cast_column(
    "audio_filepath",
    Audio(sampling_rate=16000)
)

print("The Audio Column Has Been Successfully Cast To 16kHz Sampling Rate")

train_test_split_dataset = dataset["test"].train_test_split(
    test_size=0.1,
    seed=42
)

train_validation_split_dataset = (
    train_test_split_dataset["train"]
    .train_test_split(
        test_size=0.1,
        seed=42
    )
)

dataset_train = train_validation_split_dataset["train"]
dataset_validation = train_validation_split_dataset["test"]
dataset_test = train_test_split_dataset["test"]

print("")
print("The Svarah Dataset Has Been Resampled To 16KHz And Partitioned Into The Respective Sets")

In [ ]:
# Cleaning Text Transcriptions And Constructing A Character-Level Vocabulary For The Fine-Tuning Of The Wav2Vec2 Model

import json
import os
import re

from datasets import concatenate_datasets

def clean_text(batch):

    cleaned_text = []

    for text in batch["text"]:
        text = text.lower()
        text = re.sub(r"[^a-z\s']", "", text)
        text = re.sub(r"\s+", " ", text).strip()

        cleaned_text.append(text)
        
    batch["text"] = cleaned_text
    return batch

dataset_train = dataset_train.map(clean_text, batched=True)
dataset_validation = dataset_validation.map(clean_text, batched=True)
dataset_test = dataset_test.map(clean_text, batched=True)

print("The Text Transcripts Have Been Cleaned Successfully")
print("")

combined_dataset = concatenate_datasets([dataset_train, dataset_validation, dataset_test])

def extract_all_characters(batch):
    combined_text = " ".join(batch["text"])
    vocabulary = list(set(combined_text))

    return {
        "vocab": [vocabulary],
        "all_text": [combined_text]
    }

vocabulary_dataset = combined_dataset.map(
    extract_all_characters,
    batched=True,
    batch_size=-1,
    keep_in_memory=True,
    remove_columns=combined_dataset.column_names
)

vocabulary_list = list(set(vocabulary_dataset["vocab"][0]))
vocabulary_list.sort()

vocabulary_dictionary = {
    character: index
    for index, character in enumerate(vocabulary_list)
}

vocabulary_dictionary["|"] = vocabulary_dictionary[" "]

del vocabulary_dictionary[" "]

vocabulary_dictionary["[UNK]"] = len(vocabulary_dictionary)
vocabulary_dictionary["[PAD]"] = len(vocabulary_dictionary)

print(f"The Final Size Of The Character-Level Vocabulary: {len(vocabulary_dictionary)}")
print("")

print("The Constructed Character-Level Vocabulary: ")
display(list(vocabulary_dictionary.keys()))

os.makedirs("vocabulary", exist_ok=True)

with open("vocabulary/vocabulary.json", "w") as vocabulary_file:
    json.dump(vocabulary_dictionary, vocabulary_file)

print("")
print("The Constructed Character-Level Vocabulary Has Been Successfully Saved To vocabulary/vocabulary.json")

In [ ]:
# Initializing The Wav2Vec2CTCTokenizer And Wav2Vec2FeatureExtractor And Combining Them Into A Wav2Vec2Processor

from transformers import (
    Wav2Vec2CTCTokenizer,
    Wav2Vec2FeatureExtractor,
    Wav2Vec2Processor
)

tokenizer = Wav2Vec2CTCTokenizer(
    "vocabulary/vocabulary.json",
    unk_token="[UNK]",
    pad_token="[PAD]",
    word_delimiter_token="|"
)

feature_extractor = Wav2Vec2FeatureExtractor(
    feature_size=1,
    sampling_rate=16000,
    padding_value=0.0,
    do_normalize=True,
    return_attention_mask=True
)

processor = Wav2Vec2Processor(
    feature_extractor=feature_extractor,
    tokenizer=tokenizer
)

processor.save_pretrained("vocabulary/processor")

print(f"The Sampling Rate Of The Wav2Vec2FeatureExtractor: {feature_extractor.sampling_rate} Hz")
print("The Wav2Vec2Processor Has Been Successfully Saved To vocabulary/processor")

In [ ]:
# Preparing The Training And Validation Sets Of The Svarah Dataset For The Fine-Tuning Of The Wav2Vec2 Model

def prepare_dataset(batch):
    audio = batch["audio_filepath"]
    batch["input_values"] = processor(audio["array"], sampling_rate=audio["sampling_rate"]).input_values[0]
    batch["input_length"] = len(batch["input_values"])
    batch["labels"] = processor.tokenizer(batch["text"]).input_ids

    return batch

dataset_train = dataset_train.map(
    prepare_dataset,
    remove_columns=dataset_train.column_names,
    num_proc=1
)

dataset_validation = dataset_validation.map(
    prepare_dataset,
    remove_columns=dataset_validation.column_names,
    num_proc=1
)

print("The Training And Validation Sets Of The Svarah Dataset Has Been Prepared And Mapped Successfully")

In [ ]:
# Initializing The Pretrained Wav2Vec2 Model For Connectionist Temporal Classification Fine-Tuning Using The Svarah Dataset

from transformers import Wav2Vec2ForCTC

model = Wav2Vec2ForCTC.from_pretrained(
    "facebook/wav2vec2-large-960h",
    ctc_loss_reduction="mean",
    pad_token_id=processor.tokenizer.pad_token_id,
    vocab_size=len(processor.tokenizer),
    ignore_mismatched_sizes=True
)

model.freeze_feature_encoder()

total_parameters = sum(parameter.numel() for parameter in model.parameters())
trainable_parameters = sum(parameter.numel() for parameter in model.parameters() if parameter.requires_grad)
frozen_parameters = total_parameters - trainable_parameters

print("The Wav2Vec2 Model Has Been Loaded: facebook/wav2vec2-large-960h")
print("")
print(f"The Total Number Of Parameters: {total_params:,} Parameters")
print(f"The Number Of Trainable Parameters: {trainable_params:,} Parameters")
print(f"The Number Of Frozen Parameters:    {frozen_params:,} Parameters")
print("")
print(f"The Custom Character-Level Vocabulary Size Has Been Set To: {len(processor.tokenizer)}")

In [ ]:
# Defining A Custom DataCollatorWithCTCPadding Class Responsible For Dynamic Padding During The Fine-Tuning The Wav2Vec2 Model

from dataclasses import dataclass
from typing import Dict, List, Union

import torch

@dataclass
class DataCollatorCTCWithPadding:
    processor: Wav2Vec2Processor
    padding: Union[bool, str] = True

    def __call__(self, features: List[Dict[str, Union[List[int], torch.Tensor]]]) -> Dict[str, torch.Tensor]:

        input_features = [{"input_values": feature["input_values"]} for feature in features]
        label_features = [{"input_ids": feature["labels"]} for feature in features]

        batch = self.processor.pad(
            input_features,
            padding=self.padding,
            return_tensors="pt"
        )

        labels_batch = self.processor.tokenizer.pad(
            label_features,
            padding=self.padding,
            return_tensors="pt"
        )

        labels = labels_batch["input_ids"].masked_fill(
            labels_batch.attention_mask.ne(1),
            -100
        )

        batch["labels"] = labels

        return batch


data_collator = DataCollatorCTCWithPadding(
    processor=processor,
    padding=True
)

print("A Custom DataCollatorWithCTCPadding Class Has Been Initialized Successfully")

In [ ]:
# Loading The Word Error Rate Evaluation Metric And Defining The Model Evaluation Function To Evaluate The Fine-Tuned Wav2Vec2 Model

import evaluate

wer_metric = evaluate.load("wer")

def compute_metrics(pred):
    prediction_logits = pred.predictions
    prediction_ids = np.argmax(prediction_logits, axis=-1)
    pred.label_ids[pred.label_ids == -100] = processor.tokenizer.pad_token_id

    predicted_transcriptions = processor.batch_decode(prediction_ids)
    reference_transcriptions = processor.batch_decode(pred.label_ids, group_tokens=False)
    word_error_rate = wer_metric.compute(predictions=predicted_transcriptions, references=reference_transcriptions)

    print("")
    print(f"The Sample Predicted Transcription : {predicted_transcriptions[0]}")
    print(f"The Sample Reference Transcription : {reference_transcriptions[0]}")
    print("")
    return {"wer": word_error_rate}

print("The Word Error Rate Evaluation Metric Has Been Loaded Successfully")

In [ ]:
# Configuring The Training Arguments And Hyperparameters Required For The Fine-Tuning Of The Wav2Vec2 Model

from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="./wav2vec2-svarah-finetuned-model",
    group_by_length=True,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    gradient_accumulation_steps=2,
    eval_strategy="epoch",
    save_strategy="epoch",
    num_train_epochs=30,
    fp16=True,
    learning_rate=1e-4,
    warmup_steps=500,
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="wer",
    greater_is_better=False,
    logging_steps=50,
    report_to="none",
    dataloader_num_workers=2,
)

print("The Training Arguments And Hyperparameter Configurations Have Been Initialized Successfully")

In [ ]:
# Initializing The Hugging Face Trainer And Starting The Fine-Tuning Of The Wav2Vec2 Model Using The Svarah Dataset

from transformers import Trainer

trainer = Trainer(
    model=model,
    data_collator=data_collator,
    args=training_args,
    compute_metrics=compute_metrics,
    train_dataset=dataset_train,
    eval_dataset=dataset_validation,
    processing_class=processor.feature_extractor,
)

print("The Hugging Face Trainer Has Been Initialized Successfully And Starting The Finetuning Of The Wav2Vec2 Model With The Svarah Dataset")
print("")
trainer.train()

In [ ]:
# Saving The Fine-Tuned Wav2Vec2 Model Weights, Processor Configuration, And Complete Training Artifacts For Future Inference And Deployment

import shutil
import torch

torch.save(model.state_dict(), "./wav2vec2-svarah-finetuned-model/wav2vec2_svarah_finetuned.pt")

model.save_pretrained("./wav2vec2-svarah-finetuned-model/model")
processor.save_pretrained("./wav2vec2-svarah-finetuned-model/processor")

shutil.make_archive("wav2vec2_svarah_finetuned_model", "zip", "./wav2vec2-svarah-finetuned-model")

print("The Fine-Tuned Wav2Vec2 Model Artifacts Have Been Successfully Saved And Ready For Inference And Deployment")